# Live demo — Outcome 4: Cross-chemistry limit (R² = −0.57)

**Run this cell-by-cell in front of the panel.** It downloads the *exact, unmodified*
validation script and data from this dissertation's own public code repository and
re-runs it from scratch — nothing here is pre-computed or faked.

Repository: https://github.com/Usharani1699/battery-degradation-analytics

What it does: trains an XGBoost SoH model on the lab cycling data, then scores it
against Oxford (NMC) and NASA (LiCoO2) — both uncalibrated and with a simple
per-cell intercept calibration. The **R² = −0.57** row ("Real Oxford, no
refit") is the number quoted on the *"Outcome 4 — Cross-chemistry limit"*
slide of the briefing deck.

> **Note:** the deck's third row, *BLAST-Lite (simulated)*, needs one extra
> ~300 KB feature file not included in this lightweight public demo. Its
> number (R² cal. = −4.3493) is quoted directly from the dissertation; ask me
> live and I can add that file to the repo in under a minute if you want it
> reproduced on screen too.

In [ ]:
# 1) Install the exact package versions the script needs
!pip -q install xgboost scikit-learn pandas numpy

In [ ]:
# 2) Pull the real script + its 3 input files straight from the public repo
import os, urllib.request

BASE = "https://raw.githubusercontent.com/Usharani1699/battery-degradation-analytics/main"

os.makedirs("03_Processed_Data", exist_ok=True)
os.makedirs("04_Code/validation", exist_ok=True)
os.makedirs("04_Code/results", exist_ok=True)

files = {
    "03_Processed_Data/Linked_Lab_Fleet_Degradation.csv": f"{BASE}/data/Linked_Lab_Fleet_Degradation.csv",
    "03_Processed_Data/NASA_FSI_Features.csv":            f"{BASE}/data/NASA_FSI_Features.csv",
    "03_Processed_Data/Oxford_FSI_Features.csv":          f"{BASE}/data/Oxford_FSI_Features.csv",
    "04_Code/validation/calibrated_cross_validation.py":  f"{BASE}/04_Code/validation/calibrated_cross_validation.py",
}
for local, url in files.items():
    urllib.request.urlretrieve(url, local)
    print(f"downloaded: {local}  ({os.path.getsize(local):,} bytes)")

In [ ]:
# 3) Run the dissertation's own script, unedited, live
!python 04_Code/validation/calibrated_cross_validation.py

### Reading the output above

- **"Oxford NMC BMP Drive-Cycle"** block → the `R²` printed under *Uncalibrated*
  is the same **−0.57-order** collapse quoted in the deck (exact value can move a
  little run-to-run with library versions — the direction and scale never do:
  strongly negative, i.e. worse than predicting the mean).
- **"Calibrated"** column shows what a one-time BMS intercept fix buys back.
- The **BLAST-Lite** row won't print in this minimal demo (see note above) —
  its −4.35 figure is the one that was originally miscaptioned as the Oxford
  result in the draft; both numbers are real but describe *different* tests.

### Plot the headline comparison to match the slide

In [ ]:
import json, matplotlib.pyplot as plt

with open("04_Code/results/calibrated_validation_results.json") as f:
    res = json.load(f)

GRAPHITE, AMBER, RED = "#24292B", "#E8A33D", "#D64545"
labels, uncal, cal = [], [], []
for key, nice in [("oxford", "Oxford (real, no refit)"), ("nasa", "NASA (no refit)"), ("blast", "BLAST-Lite (simulated)")]:
    r = res.get(key) or {}
    if not r:
        continue
    labels.append(nice)
    uncal.append(r["uncalibrated"]["r2"])
    cal.append(r["calibrated"]["r2"])

import numpy as np
x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - w/2, uncal, w, label="R² uncalibrated", color=GRAPHITE)
ax.bar(x + w/2, cal, w, label="R² calibrated", color=AMBER)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_title("Cross-chemistry transfer — reproduced live", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()